# Cross-Sectional Return Distributions

Equity returns do not follow a normal distribution — and their cross-sectional structure changes dramatically over time.  This notebook visualises cross-sectional distributions from a synthetic factor-driven panel and documents their non-Gaussian, time-varying character.


In [1]:
import sys; sys.path.insert(0, "..")
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import norm, kurtosis, skew

from data.synthetic import generate_factor_driven_panel, generate_regime_switching_panel
from otreturns.distributions import EmpiricalDistribution, DistributionPanel
from otreturns.distances import wasserstein_1d
print("Imports OK")


Imports OK


## 1. Load Synthetic Data

In [2]:
panel_df, factor_returns, factor_loadings = generate_factor_driven_panel(
    n_dates=120, n_stocks=500, n_factors=3, seed=42
)
panel = DistributionPanel.from_panel(panel_df, min_stocks=50, winsorize=0.005)
dates = panel.dates
print(f"Panel: {len(dates)} dates, {panel[dates[0]].n} stocks per date (approx)")
print(f"Date range: {dates[0]} … {dates[-1]}")


Panel: 120 dates, 500 stocks per date (approx)
Date range: 2018-01-02 00:00:00 … 2018-06-18 00:00:00


## 2. Time-Varying Cross-Sectional Statistics

In [3]:
stats = []
for d in dates:
    dist = panel[d]
    m = dist.moments(order=4)
    stats.append({
        "date": d,
        "mean":     m["mean"],
        "std":      np.sqrt(m["variance"]),
        "skewness": m["skewness"],
        "kurtosis": m["kurtosis"],
    })
df_stats = pd.DataFrame(stats)

fig, axes = plt.subplots(4, 1, figsize=(12, 10), sharex=True)
labels = ["Mean", "Std dev", "Skewness", "Excess kurtosis"]
cols   = ["mean", "std", "skewness", "kurtosis"]
colors = ["steelblue", "darkorange", "forestgreen", "crimson"]

for ax, col, label, color in zip(axes, cols, labels, colors):
    ax.plot(range(len(dates)), df_stats[col], color=color, lw=1.5)
    ax.set_ylabel(label)
    ax.axhline(0, color="black", lw=0.5, ls="--")
    ax.grid(alpha=0.3)

axes[-1].set_xlabel("Date index")
axes[0].set_title("Cross-sectional return distribution statistics over time")
plt.tight_layout()
plt.savefig("../figures/cs_stats_over_time.png", dpi=100)
plt.show()
print(df_stats.describe().round(5).to_string())


                      date       mean        std   skewness   kurtosis
count                  120  120.00000  120.00000  120.00000  120.00000
mean   2018-03-25 21:36:00    0.00004    0.01181    0.01395   -0.16112
min    2018-01-02 00:00:00   -0.00096    0.00495   -0.36645   -0.57864
25%    2018-02-12 18:00:00   -0.00020    0.00781   -0.05633   -0.28952
50%    2018-03-26 12:00:00    0.00006    0.01039    0.01774   -0.18243
75%    2018-05-07 06:00:00    0.00025    0.01472    0.07146   -0.01657
max    2018-06-18 00:00:00    0.00077    0.03112    0.37914    0.43041
std                    NaN    0.00036    0.00524    0.12629    0.19844


## 3. Non-Gaussianity

The cross-section is **not** Gaussian:
- **Fat tails** (excess kurtosis > 0) from factor co-movement.
- **Skewness** varies with the factor structure.
- The distribution evolves over time — standard return model assumptions fail.


In [4]:
# Compare a single cross-section to a Gaussian with the same mean/std
d = panel[dates[60]]
m = d.moments(2)
mean_d = m["mean"]; std_d = np.sqrt(m["variance"])

x = np.linspace(mean_d - 4*std_d, mean_d + 4*std_d, 300)
bw = 1.06 * std_d * d.n**(-0.2)

def kde(samples, x_grid, bw_):
    diff = (x_grid[:, None] - samples[None, :]) / bw_
    return np.mean(np.exp(-0.5 * diff**2), axis=1) / (bw_ * np.sqrt(2*np.pi))

dens_empirical = kde(d.samples, x, bw)
dens_gaussian  = norm.pdf(x, mean_d, std_d)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].fill_between(x, dens_empirical, alpha=0.3, color="steelblue")
axes[0].plot(x, dens_empirical, color="steelblue", lw=2, label="Empirical")
axes[0].plot(x, dens_gaussian,  color="red",       lw=2, ls="--", label="Gaussian fit")
axes[0].legend(); axes[0].set_xlabel("Return"); axes[0].set_ylabel("Density")
axes[0].set_title(f"Cross-section at date {dates[60]}"); axes[0].grid(alpha=0.3)

# Q-Q plot against Normal
sorted_q = np.sort(d.samples)
theoretical_q = norm.ppf(np.linspace(0.01, 0.99, len(sorted_q)), mean_d, std_d)
axes[1].scatter(theoretical_q, sorted_q, s=2, color="steelblue", alpha=0.5)
lo = min(theoretical_q.min(), sorted_q.min())
hi = max(theoretical_q.max(), sorted_q.max())
axes[1].plot([lo, hi], [lo, hi], "r--", lw=1.5, label="Normal")
axes[1].set_xlabel("Theoretical quantile"); axes[1].set_ylabel("Empirical quantile")
axes[1].set_title("Q-Q plot vs Normal"); axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig("../figures/non_gaussianity.png", dpi=100)
plt.show()
m4 = d.moments(4)
print(f"Skewness: {m4['skewness']:.4f}  |  Excess kurtosis: {m4['kurtosis']:.4f}")


Skewness: 0.1068  |  Excess kurtosis: -0.0184


## 4. Wasserstein Distance Time Series

The Wasserstein distance between consecutive cross-sections measures the "speed" of distributional change.  Spikes in W₂(μₜ, μₜ₋₁) indicate regime transitions.


In [5]:
w2_consec = [wasserstein_1d(panel[dates[t-1]], panel[dates[t]]) for t in range(1, len(dates))]

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(range(1, len(dates)), w2_consec, color="steelblue", lw=1.2, alpha=0.8)
ax.fill_between(range(1, len(dates)), w2_consec, alpha=0.2, color="steelblue")
ax.axhline(np.mean(w2_consec), color="red", ls="--", lw=1.5,
           label=f"Mean = {np.mean(w2_consec):.5f}")
ax.set_xlabel("Date index"); ax.set_ylabel("W₂(μₜ₋₁, μₜ)")
ax.set_title("Consecutive Wasserstein distance — speed of distributional change")
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("../figures/w2_time_series.png", dpi=100)
plt.show()
print(f"Mean W₂:   {np.mean(w2_consec):.6f}")
print(f"Max  W₂:   {np.max(w2_consec):.6f}  at t={np.argmax(w2_consec)+1}")


Mean W₂:   0.005245
Max  W₂:   0.017309  at t=98


## 5. Distribution Heatmap

Visualising the quantile functions over time reveals the evolving structure of the cross-section.  A heatmap of $F_t^{-1}(u)$ as a function of both time and quantile level is a compact summary.


In [6]:
u_grid = np.linspace(0.05, 0.95, 80)
Q_mat = np.array([panel[d].quantile_function(u_grid) for d in dates])  # (T, M)

fig, ax = plt.subplots(figsize=(12, 5))
im = ax.imshow(Q_mat.T, aspect="auto", origin="lower", cmap="RdYlGn_r",
               extent=[0, len(dates), u_grid[0], u_grid[-1]])
plt.colorbar(im, ax=ax, label="Return quantile")
ax.set_xlabel("Date index"); ax.set_ylabel("Quantile level u")
ax.set_title("Cross-sectional quantile function heatmap  F⁻¹(u, t)")
plt.tight_layout()
plt.savefig("../figures/quantile_heatmap.png", dpi=100)
plt.show()
print("Done.")


Done.
